# Train the YOLO emitter detector on Colab (GPU)

Trains `src/detect/model.py` (Chua's single-class detector: one box per transmitter) on synthetic scenes from our own generators. The 60-class classifier names each box later (`src/pipeline/`), so this notebook trains **only the detector**.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

**Where everything is saved (Google Drive, so a disconnect loses nothing):**

| What | Drive folder | Put it on your PC at |
|---|---|---|
| Trained detector `emitter_detector.pt`, `history.json`, `eval.json`, PNGs | `FOI26/detector_results/<RUN_NAME>/` | `<repo>/results/detector/` |
| Scenes `train.npz`, `val.npz`, `test.npz` | `FOI26/detector_data/scenes_<N_TRAIN>/` | `<repo>/data/detect/` (optional) |

The code is cloned to `/content` (not kept). Its `results/detector` folder points at the Drive results folder, so every script writes there with its normal default paths, after every epoch. Scenes are made on the fast Colab disk, then copied to Drive once; a rerun copies them back instead of making them again.

**Run:** first `QUICK = True` (4,000 scenes x 5 epochs, about 15 min) to check everything works, then `QUICK = False` for the real run.

**If Colab disconnects:** run all cells again with the same settings. The scenes come back from Drive; training restarts from epoch 1 (the detector script has no resume), and the best model so far stays on Drive until a better epoch replaces it.

## 1. Settings

In [ ]:
QUICK = True        # True: 4,000 scenes x 5 epochs, to check everything runs. False: the real run.

if QUICK:
    N_TRAIN, N_VAL, N_TEST, EPOCHS = 4000, 500, 500, 5
else:
    N_TRAIN, N_VAL, N_TEST, EPOCHS = 30000, 5000, 5000, 30
RUN_NAME = f'2026-11-05_detector_{N_TRAIN}x{EPOCHS}'   # one Drive folder per run
BRANCH = 'main'

FOI26 = '/content/drive/MyDrive/sedic/FOI26'               # shared folder (add a shortcut to My Drive if it was shared with you)
DRIVE_SCENES = f'{FOI26}/detector_data/scenes_{N_TRAIN}'   # train.npz val.npz test.npz
DRIVE_OUT    = f'{FOI26}/detector_results/{RUN_NAME}'      # emitter_detector.pt history.json eval.json *.png
CODE = '/content/sedicAI_NEXA'
LOCAL_SCENES = '/content/detect'                           # fast local copy used for training
print(RUN_NAME, '|', N_TRAIN, N_VAL, N_TEST, 'scenes,', EPOCHS, 'epochs')

## 2. GPU check

In [ ]:
import torch, os
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: Runtime -> Change runtime type -> T4 GPU')
print('CPU cores:', os.cpu_count())

## 3. Mount Drive, get the code, point its folders at Drive and the local disk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
assert os.path.isdir(FOI26), f'{FOI26} not found: add the shared FOI26 folder to My Drive, or fix FOI26 in cell 1'

%cd /content
!rm -rf {CODE}
!git clone -q -b {BRANCH} https://github.com/eavan127/sedicAI_NEXA.git {CODE}
%cd {CODE}
!git log -1 --format='commit %h  %s'
!pip install -q pyyaml
assert 'device' in open('scripts/train_detector.py').read(), 'this commit has no GPU support in train_detector.py'

os.makedirs(DRIVE_OUT, exist_ok=True); os.makedirs(DRIVE_SCENES, exist_ok=True); os.makedirs(LOCAL_SCENES, exist_ok=True)
os.makedirs(f'{CODE}/data', exist_ok=True); os.makedirs(f'{CODE}/results', exist_ok=True)
for link, target in ((f'{CODE}/data/detect', LOCAL_SCENES), (f'{CODE}/results/detector', DRIVE_OUT)):
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(target, link)
    print(link, '->', target)

## 4. Scenes: copy back from Drive if they exist, otherwise make them and save to Drive

Waterfall pictures (10.24 ms x 3.2 MHz) with 0-6 transmitters each, drawn by our own generators; boxes are computed automatically, no hand labelling. Making 30,000 takes roughly 30 min on Colab's 2 CPU cores (4,000: a few minutes).

In [ ]:
files = ('train.npz', 'val.npz', 'test.npz')
if all(os.path.exists(f'{DRIVE_SCENES}/{f}') for f in files):
    for f in files:
        shutil.copy(f'{DRIVE_SCENES}/{f}', f'{LOCAL_SCENES}/{f}')
    print('scenes copied back from Drive')
else:
    workers = os.cpu_count()
    !python scripts/make_scenes.py --n-train {N_TRAIN} --n-val {N_VAL} --n-test {N_TEST} --workers {workers}
    for f in files:
        shutil.copy(f'{LOCAL_SCENES}/{f}', f'{DRIVE_SCENES}/{f}')
    print('scenes saved to', DRIVE_SCENES)
!ls -lh {LOCAL_SCENES}

## 5. Look at the training pictures (white boxes = truth)

In [ ]:
from IPython.display import Image, display
!python scripts/show_scenes.py --split train --n 8 --no-open --out {DRIVE_OUT}/train_scenes.png
display(Image(f'{DRIVE_OUT}/train_scenes.png'))

## 6. Train

The first line must say **GPU Tesla T4**. One line per epoch: seconds per epoch x epochs = total time. The best epoch (validation AP@0.5) is saved to Drive as `emitter_detector.pt` straight away.

In [ ]:
!python scripts/train_detector.py --epochs {EPOCHS}
!ls -lh {DRIVE_OUT}

## 7. Training curve

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f'{DRIVE_OUT}/history.json'))
ep = [e['epoch'] for e in h]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(ep, [e['loss'] for e in h], 'o-'); ax[0].set_title('training loss'); ax[0].set_xlabel('epoch')
ax[1].plot(ep, [e['val_ap50'] for e in h], 'o-', color='tab:green'); ax[1].set_title('validation AP@0.5 (higher = better)'); ax[1].set_xlabel('epoch')
fig.tight_layout(); fig.savefig(f'{DRIVE_OUT}/training_curve.png', dpi=100); plt.show()
best = max(h, key=lambda e: e['val_ap50'])
print(f"best val AP@0.5 {best['val_ap50']:.3f} at epoch {best['epoch']}")

## 8. Score on test scenes it never saw

YOLO against the classical loudness (energy) detector, each at its best threshold chosen on validation. Writes `eval.json` and `eval_examples.png` to Drive.

In [ ]:
!python scripts/eval_detector.py
display(Image(f'{DRIVE_OUT}/eval_examples.png'))

## 9. What YOLO finds (white = truth, red = YOLO)

In [ ]:
!python scripts/show_scenes.py --split test --n 8 --model --no-open --out {DRIVE_OUT}/test_found.png
display(Image(f'{DRIVE_OUT}/test_found.png'))
!ls -lh {DRIVE_OUT}
print('Done. On your PC: copy the Drive folder', DRIVE_OUT, 'into <repo>/results/detector/')